# Project 03 — E-commerce Sales Trend Analysis

**Goal:** engineer date features, build pivot summaries, identify customer and category patterns, and write recommendations tied to the numbers.

**Data note:** the included 2,400-row order file is deterministic synthetic data generated for this project. Customer IDs and transactions are fictional; there is no real customer information.

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

DATA_PATH = Path("data/ecommerce_sales_generated.csv")
df = pd.read_csv(DATA_PATH, parse_dates=["order_date"])
df["month"] = df["order_date"].dt.to_period("M").astype(str)
df["day_of_week"] = df["order_date"].dt.day_name()
df["year"] = df["order_date"].dt.year
df["month_number"] = df["order_date"].dt.month
print(f"Rows: {len(df):,} | unique orders: {df['order_id'].nunique():,} | unique fictional customers: {df['customer_id'].nunique():,}")
display(df.head())
print("Duplicate order IDs:", int(df["order_id"].duplicated().sum()))

Rows: 2,400 | unique orders: 2,400 | unique fictional customers: 283
    order_id order_date customer_id  ... day_of_week  year month_number
0  ORD-00001 2025-04-17   CUST-0015  ...    Thursday  2025            4
1  ORD-00002 2025-10-18   CUST-0002  ...    Saturday  2025           10
2  ORD-00003 2025-08-10   CUST-0004  ...      Sunday  2025            8
3  ORD-00004 2025-06-29   CUST-0011  ...      Sunday  2025            6
4  ORD-00005 2025-06-25   CUST-0004  ...   Wednesday  2025            6

[5 rows x 14 columns]
Duplicate order IDs: 0


## 1. Monthly revenue trend

In [2]:
monthly = df.pivot_table(index="month", values="revenue", aggfunc="sum", observed=False).reset_index()
monthly["month_start"] = pd.PeriodIndex(monthly["month"], freq="M").to_timestamp()
display(monthly[["month", "revenue"]].style.format({"revenue": "${:,.0f}"}))
sns.set_theme(style="whitegrid", context="notebook")
OUT = Path("charts")
OUT.mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(9, 4.8))
sns.lineplot(data=monthly, x="month_start", y="revenue", marker="o", color="#2878B5", linewidth=2.5, ax=ax)
ax.set(title="Monthly revenue rises into the year-end in this simulated sample", xlabel="Month", ylabel="Revenue (simulated currency units)")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(OUT / "01_monthly_revenue.png", dpi=180, bbox_inches="tight")
plt.show()

C:\Users\gana6\Documents\Codex\2026-11-12\referenced-chatgpt-conversation-this-is-an\outputs\InternCircle_Data_Science_Projects\project_03_sales_trends\sales_trend_analysis.ipynb:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ]


## 2. Category mix and customer segments

In [3]:
category_summary = df.pivot_table(index="category", values=["revenue", "order_id"],
                                  aggfunc={"revenue": "sum", "order_id": "count"}, observed=False)
category_summary = category_summary.rename(columns={"order_id": "orders"}).sort_values("revenue", ascending=False)
display(category_summary.style.format({"revenue": "${:,.0f}", "orders": "{:,.0f}"}))

fig, ax = plt.subplots(figsize=(8, 4.8))
sns.barplot(data=category_summary.reset_index(), y="category", x="revenue", color="#4F9D69", ax=ax)
ax.set(title="Category revenue is concentrated in the largest categories", xlabel="Revenue (simulated currency units)", ylabel="Category")
fig.tight_layout()
fig.savefig(OUT / "02_revenue_by_category.png", dpi=180, bbox_inches="tight")
plt.show()

customers = df.groupby("customer_id", as_index=False).agg(orders=("order_id", "count"), revenue=("revenue", "sum"))
customers["revenue_rank_pct"] = customers["revenue"].rank(method="first", pct=True)
customers["segment"] = "Lower 50%"
customers.loc[customers["revenue_rank_pct"] > .50, "segment"] = "Middle 40%"
customers.loc[customers["revenue_rank_pct"] > .90, "segment"] = "Top 10%"
segment_summary = customers.groupby("segment", as_index=False).agg(customers=("customer_id", "count"), revenue=("revenue", "sum"))
segment_summary["revenue_share"] = segment_summary["revenue"] / segment_summary["revenue"].sum()
display(segment_summary.style.format({"revenue": "${:,.0f}", "revenue_share": "{:.1%}"}))

C:\Users\gana6\Documents\Codex\2026-11-12\referenced-chatgpt-conversation-this-is-an\outputs\InternCircle_Data_Science_Projects\project_03_sales_trends\sales_trend_analysis.ipynb:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  "\n",


## 3. Top-customer concentration and recommendations

In [4]:
import numpy as np
ranked = customers.sort_values("revenue", ascending=False).reset_index(drop=True)
ranked["cumulative_revenue_share"] = ranked["revenue"].cumsum() / ranked["revenue"].sum()
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(np.arange(1, len(ranked) + 1), ranked["cumulative_revenue_share"], color="#7B61A8", linewidth=2.5)
ax.axhline(.80, color="#D17B0F", linestyle="--", linewidth=1.2, label="80% of revenue")
ax.set(title="Cumulative revenue highlights customer concentration", xlabel="Customers ranked from highest to lowest spend", ylabel="Cumulative share of revenue", ylim=(0, 1.02))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "03_customer_revenue_concentration.png", dpi=180, bbox_inches="tight")
plt.show()

total_revenue = float(df["revenue"].sum())
peak_month = monthly.loc[monthly["revenue"].idxmax()]
peak_category = category_summary.index[0]
top10_share = float(customers.loc[customers["segment"].eq("Top 10%"), "revenue"].sum() / total_revenue)
print(f"Total simulated revenue: ${total_revenue:,.2f}")
print(f"Peak month: {peak_month['month']} (${peak_month['revenue']:,.2f})")
print(f"Largest category: {peak_category} (${category_summary.loc[peak_category, 'revenue']:,.2f})")
print(f"Top 10% of customers contribute {top10_share:.1%} of revenue")

C:\Users\gana6\Documents\Codex\2026-11-12\referenced-chatgpt-conversation-this-is-an\outputs\InternCircle_Data_Science_Projects\project_03_sales_trends\sales_trend_analysis.ipynb:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  "**Data note:** the included 2,400-row order file is deterministic synthetic data generated for this project. Customer IDs and transactions are fictional; there is no real customer information."
Total simulated revenue: $342,870.53
Peak month: 2025-12 ($72,776.91)
Largest category: Electronics ($180,580.57)
Top 10% of customers contribute 72.5% of revenue


### Recommendations

1. Compare the year-end peak with fulfillment capacity before increasing seasonal inventory; this pattern is simulated and is not a forecast.
2. Test category-specific promotions for the largest category, then compare margin as well as revenue before scaling.
3. Trial a retention offer for the top-spend segment, while monitoring whether the offer adds incremental purchases rather than discounting purchases that would happen anyway.

The data is simulated; these recommendations are examples of how to turn the calculated summaries into cautious business actions.